# 2026 approach stats by lineup slot

Clustering found no discrete hitter archetypes (see `cluster_hitters.ipynb`) — the five approach
stats form a continuum. So instead of inventing classes, we let **real MLB lineup usage define the
template**: tag each qualified hitter with their most-frequent starting slot (from
`pull_lineup_data.py`), group the approach stats by slot, and look at the per-slot profiles.

**Exploration only.** The goal here is to *see* the slot profiles and judge how distinct they are —
not yet to build a classifier. What we do with these numbers is decided afterward.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")

STATS = ["k_pct", "bb_pct", "chase_pct", "whiff_pct", "barrel_pct"]
LABELS = {"k_pct": "Strikeout %", "bb_pct": "Walk %", "chase_pct": "Chase %",
          "whiff_pct": "Whiff %", "barrel_pct": "Barrel %"}

# Approach stats (PA >= 150, the established analysis population) + each hitter's modal slot.
stats = pd.read_csv("hitter_stats_2026.csv")
stats = stats[stats["pa"] >= 150].copy()
lineup = pd.read_csv("hitter_lineup_2026.csv")

df = stats.merge(lineup[["batter", "primary_slot", "games_started", "slot_share"]], on="batter", how="inner")
print(f"{len(df)} qualified hitters matched to a lineup slot (of {len(stats)} PA>=150)")
df[["name", "pa", "primary_slot", "slot_share"] + STATS].head()

## Slot profiles: mean of each stat by lineup slot

The "prototype" for each of the 9 spots — what the average hitter batting there actually looks like.

In [ ]:
profile = df.groupby("primary_slot")[STATS].mean().round(1)
profile["n"] = df.groupby("primary_slot").size()
profile

## Each stat across the batting order

One panel per stat. If lineups are built the traditional way we'd expect barrel% and K% to bulge in
the 3-5 "heart" of the order, walk% to lean higher up, and the bottom (6-9) to be weaker — though the
modern #2 spot is often a star, so the gradient may not be a clean ramp.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, stat in zip(axes.ravel(), STATS):
    ax.bar(profile.index, profile[stat], color="steelblue")
    ax.set_title(LABELS[stat]); ax.set_xlabel("lineup slot"); ax.set_xticks(range(1, 10))
axes.ravel()[-1].axis("off")
fig.suptitle("Mean approach stat by lineup slot (2026, PA>=150)", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## Standardized profile heatmap

Each stat z-scored across the 9 slot means, so we can compare *shapes* on one scale (red = high for
that stat, blue = low). Makes the power gradient vs. the flat stats obvious at a glance.

In [ ]:
z = pd.DataFrame(StandardScaler().fit_transform(profile[STATS]),
                 index=profile.index, columns=[LABELS[s] for s in STATS])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(z, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            cbar_kws={"label": "z-score across slots"})
ax.set_ylabel("lineup slot"); ax.set_title("Slot profiles (standardized per stat)")
plt.tight_layout(); plt.show()

## Spread within each slot

Means can hide a lot. These boxplots show the full distribution of each stat within a slot — if the
boxes overlap heavily across slots, the slot "prototype" is a weak descriptor of any individual hitter.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, stat in zip(axes.ravel(), STATS):
    sns.boxplot(data=df, x="primary_slot", y=stat, ax=ax, color="lightsteelblue")
    ax.set_title(LABELS[stat]); ax.set_xlabel("lineup slot"); ax.set_ylabel("")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

## How distinct are the slots, really?

Two quick reads per stat:
- **eta²** = share of that stat's variance explained by which slot a hitter bats in (0 = slot tells you
  nothing, 1 = slot determines the stat). Plus a one-way ANOVA p-value across slots.
- **Spearman r** of the stat vs. slot number = strength/direction of a monotonic gradient down the order.

And one overall check: the **silhouette** of the 9 slot labels on the standardized 5-stat space — the
same metric that was a weak ~0.28 for k-means. If it is near zero, slots overlap as much as the
clusters did.

In [ ]:
rows = []
for stat in STATS:
    groups = [g[stat].values for _, g in df.groupby("primary_slot")]
    grand = df[stat].mean()
    ss_between = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    ss_total = ((df[stat] - grand) ** 2).sum()
    eta2 = ss_between / ss_total
    f, p = f_oneway(*groups)
    rho, _ = spearmanr(df["primary_slot"], df[stat])
    rows.append([LABELS[stat], round(eta2, 3), round(p, 4), round(rho, 3)])
sep = pd.DataFrame(rows, columns=["stat", "eta2 (var expl. by slot)", "ANOVA p", "Spearman r vs slot"])
print(sep.to_string(index=False))

Xz = StandardScaler().fit_transform(df[STATS])
sil = silhouette_score(Xz, df["primary_slot"])
print(f"\nsilhouette of 9 slot labels on standardized 5-stat space: {sil:.3f}")
print("(k-means best was ~0.28 at k=2; near 0 => slots overlap as much as clusters did)")

## Inspect: who lands where

Sanity-check the extremes against real roles, and look at the highest-barrel hitters' slots — this is
the crux of the project's hypothesis (are the big bats concentrated in 3-5?).

In [ ]:
print("Top 15 barrel% hitters and where they bat:")
print(df.nlargest(15, "barrel_pct")[["name", "primary_slot", "slot_share", "barrel_pct", "k_pct", "bb_pct"]].to_string(index=False))

print("\nSpot checks (name -> modal slot):")
for nm in ["Aaron Judge", "Shohei Ohtani", "Nico Hoerner", "Kyle Schwarber", "Pete Alonso", "Bo Bichette"]:
    r = df[df["name"] == nm]
    if len(r):
        print(f"  {nm:16} -> slot {int(r['primary_slot'].iloc[0])} ({r['slot_share'].iloc[0]:.0%} of starts)")

## Takeaways

_(fill in after running — the step-5 decision hinges on this)_

- Which stats show a real gradient down the order vs. which are flat?
- Are the slots distinct enough to treat as 9 prototypes, or should adjacent spots be grouped
  (e.g. top / heart / bottom)?
- Does the barrel gradient support or complicate the "power belongs higher" hypothesis?